# 04. 임베딩 모델 학습 (Embedding Model)

**실행 환경:** Kaggle GPU (P100)  
**목적:** 유사 민원 검색용 한국어 Sentence Embedding 모델 파인튜닝

---

## 핵심 전략

| 항목 | 설정 |
|------|------|
| 베이스 모델 | ko-sroberta-multitask |
| 학습 데이터 | QA 쌍 17.6만건 (질문↔답변) |
| Loss 함수 | MultipleNegativesRankingLoss |
| 목적 | RAG 검색 최적화 |

In [1]:
# 패키지 설치
!pip install -q sentence-transformers datasets

In [2]:
import torch
import pandas as pd
import numpy as np
import json
from pathlib import Path
from tqdm import tqdm
from sentence_transformers import (
    SentenceTransformer,
    InputExample,
    losses,
    evaluation
)
from torch.utils.data import DataLoader
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')

# GPU 확인
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

2026-01-11 08:00:58.820924: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1768118459.020301      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1768118459.080007      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1768118459.566181      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1768118459.566236      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1768118459.566239      55 computation_placer.cc:177] computation placer alr

Device: cuda
GPU: Tesla P100-PCIE-16GB
VRAM: 17.1 GB


In [3]:
# 경로 설정
DATA_PATH = Path('/kaggle/input/kobert')
OUTPUT_PATH = Path('/kaggle/working/models/embedding')
OUTPUT_PATH.mkdir(parents=True, exist_ok=True)

# 파일 존재 확인
print("=== 데이터 파일 확인 ===")
required_files = ['qa_pairs.parquet']
for f in required_files:
    path = DATA_PATH / f
    if path.exists():
        print(f"✅ {f}")
    else:
        print(f"❌ {f} - 파일 없음!")

# 전체 파일 목록
print(f"\n전체 파일 목록:")
for f in DATA_PATH.iterdir():
    print(f"  {f.name}")

=== 데이터 파일 확인 ===
✅ qa_pairs.parquet

전체 파일 목록:
  qa_documents.json
  train_llm.parquet
  label_mapping.json
  label_encoders.joblib
  train_classification.parquet
  test_classification.parquet
  qa_pairs.parquet
  train_llm.jsonl
  val_llm.parquet
  val_classification.parquet


---
## 1. 베이스 모델 선택

In [4]:
# 한국어 임베딩 모델 - 검증된 모델 사용
BASE_MODEL = 'jhgan/ko-sroberta-multitask'

# 모델 로드
model = SentenceTransformer(BASE_MODEL, device=device)

print(f"✅ 모델 로드 완료: {BASE_MODEL}")
print(f"   임베딩 차원: {model.get_sentence_embedding_dimension()}")

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/744 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/442M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/156 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ 모델 로드 완료: jhgan/ko-sroberta-multitask
   임베딩 차원: 768


In [5]:
# 파인튜닝 전 성능 테스트
test_pairs = [
    ("인터넷뱅킹 로그인이 안돼요", "인터넷뱅킹 접속 오류 시 브라우저 캐시를 삭제하고 다시 시도해주세요."),
    ("비밀번호 5회 오류", "비밀번호 5회 오류 시 본인인증 후 재설정이 필요합니다."),
    ("카드 분실 신고", "카드 분실 시 즉시 고객센터로 연락하여 정지 요청하세요."),
    ("코로나 검사 장소", "가까운 보건소 또는 선별진료소에서 검사 가능합니다."),
]

print("=== 파인튜닝 전 Q-A 유사도 ===")
baseline_scores = []
for q, a in test_pairs:
    emb_q = model.encode(q, convert_to_tensor=True)
    emb_a = model.encode(a, convert_to_tensor=True)
    sim = torch.cosine_similarity(emb_q.unsqueeze(0), emb_a.unsqueeze(0)).item()
    baseline_scores.append(sim)
    print(f"  {q[:25]}... → {sim:.3f}")

print(f"\n평균 유사도: {np.mean(baseline_scores):.3f}")

=== 파인튜닝 전 Q-A 유사도 ===
  인터넷뱅킹 로그인이 안돼요... → 0.579
  비밀번호 5회 오류... → 0.692
  카드 분실 신고... → 0.625
  코로나 검사 장소... → 0.504

평균 유사도: 0.600


---
## 2. QA 쌍 데이터 로드

In [6]:
# QA 쌍 데이터 로드
qa_df = pd.read_parquet(DATA_PATH / 'qa_pairs.parquet')

print(f"✅ QA 쌍 로드: {len(qa_df):,}건")
print(f"\n컬럼: {qa_df.columns.tolist()}")
print(f"\n도메인 분포:")
print(qa_df['domain'].value_counts())

✅ QA 쌍 로드: 57,402건

컬럼: ['question', 'answer', 'domain', 'category', 'dialogue_id']

도메인 분포:
domain
질병관리본부    20671
금융/보험     16090
K쇼핑       11542
다산콜센터      9099
Name: count, dtype: int64


In [7]:
# 데이터 품질 확인
print("=== 데이터 품질 확인 ===")

# 텍스트 길이 분포
qa_df['q_len'] = qa_df['question'].astype(str).str.len()
qa_df['a_len'] = qa_df['answer'].astype(str).str.len()

print(f"질문 길이: 평균 {qa_df['q_len'].mean():.0f}, 최소 {qa_df['q_len'].min()}, 최대 {qa_df['q_len'].max()}")
print(f"답변 길이: 평균 {qa_df['a_len'].mean():.0f}, 최소 {qa_df['a_len'].min()}, 최대 {qa_df['a_len'].max()}")

# 유효 데이터 필터링
valid_mask = (qa_df['q_len'] >= 5) & (qa_df['a_len'] >= 10)
qa_df_valid = qa_df[valid_mask].copy()
print(f"\n유효 데이터: {len(qa_df_valid):,}건 ({len(qa_df_valid)/len(qa_df)*100:.1f}%)")

=== 데이터 품질 확인 ===
질문 길이: 평균 25, 최소 2, 최대 190
답변 길이: 평균 17, 최소 1, 최대 2028

유효 데이터: 37,842건 (65.9%)


In [8]:
# 샘플 확인
print("=== QA 쌍 샘플 ===")
for i, row in qa_df_valid.sample(3, random_state=42).iterrows():
    print(f"\n[{row['domain']}]")
    print(f"Q: {row['question'][:80]}")
    print(f"A: {row['answer'][:80]}")

=== QA 쌍 샘플 ===

[질병관리본부]
Q: 코로나에 관련하여정보를 찾고 싶습니다.
A: 마스크 재고 정보에요.

[K쇼핑]
Q: 네 수고하십니다. 지금 방송 나오는거 주문하려고 하는데요.
A: 안녕하십니까. 쇼핑 입니다.

[금융/보험]
Q: 가입한 보험을 해약하고 싶은데 어떻게 해야하나요?
A: 네.계약자 본인입니까?


---
## 3. Train/Val 분할 및 학습 데이터 준비

In [9]:
# Train/Val 분할 (90/10)
train_qa, val_qa = train_test_split(
    qa_df_valid, 
    test_size=0.1, 
    random_state=42,
    stratify=qa_df_valid['domain']  # 도메인 비율 유지
)

print(f"Train: {len(train_qa):,}건")
print(f"Val: {len(val_qa):,}건")

Train: 34,057건
Val: 3,785건


In [10]:
def create_training_pairs(df):
    """
    QA 쌍으로 학습 데이터 생성
    MultipleNegativesRankingLoss용: (question, answer) 쌍
    """
    pairs = []
    
    for _, row in tqdm(df.iterrows(), total=len(df), desc='Creating pairs'):
        q = str(row['question']).strip()
        a = str(row['answer']).strip()
        
        if len(q) >= 5 and len(a) >= 10:
            pairs.append(InputExample(texts=[q, a]))
    
    return pairs

# 전체 학습 데이터 생성 (17.6만건 전부 사용)
train_pairs = create_training_pairs(train_qa)
print(f"\n✅ 학습 쌍: {len(train_pairs):,}건")

Creating pairs: 100%|██████████| 34057/34057 [00:01<00:00, 20968.21it/s]


✅ 학습 쌍: 34,057건


In [11]:
# 평가 데이터 생성 (유사 + 비유사 쌍)
def create_eval_data(df, num_samples=2000):
    """
    평가용 데이터 생성
    - Positive: 실제 QA 쌍
    - Negative: 다른 QA의 답변과 매칭
    """
    sample = df.sample(n=min(num_samples, len(df)), random_state=42)
    
    sentences1 = []
    sentences2 = []
    scores = []
    
    questions = sample['question'].tolist()
    answers = sample['answer'].tolist()
    
    for i in range(len(questions)):
        # Positive pair (실제 QA)
        sentences1.append(questions[i])
        sentences2.append(answers[i])
        scores.append(1.0)
        
        # Negative pair (다른 답변과 매칭)
        neg_idx = (i + 1) % len(answers)  # 다음 답변 사용
        sentences1.append(questions[i])
        sentences2.append(answers[neg_idx])
        scores.append(0.0)
    
    return evaluation.EmbeddingSimilarityEvaluator(
        sentences1, sentences2, scores,
        name='qa-eval',
        show_progress_bar=True
    )

evaluator = create_eval_data(val_qa, num_samples=2000)
print("✅ 평가 데이터 생성 완료 (유사 2000 + 비유사 2000)")

✅ 평가 데이터 생성 완료 (유사 2000 + 비유사 2000)


---
## 4. 모델 파인튜닝

**MultipleNegativesRankingLoss:**
- 배치 내 다른 샘플을 자동으로 negative로 사용
- 배치 크기가 클수록 더 많은 negative → 성능 향상
- 별도 negative 생성 불필요

In [12]:
# 학습 설정
BATCH_SIZE = 64
NUM_EPOCHS = 3

# DataLoader
train_dataloader = DataLoader(
    train_pairs, 
    shuffle=True, 
    batch_size=BATCH_SIZE,
    num_workers=0,
    pin_memory=True
)

# Loss 함수
train_loss = losses.MultipleNegativesRankingLoss(model)

# Warmup
total_steps = len(train_dataloader) * NUM_EPOCHS
warmup_steps = int(total_steps * 0.1)

print("=== 학습 설정 ===")
print(f"Batch size: {BATCH_SIZE}")
print(f"Epochs: {NUM_EPOCHS}")
print(f"Total steps: {total_steps:,}")
print(f"Warmup steps: {warmup_steps:,}")
print(f"Loss: MultipleNegativesRankingLoss")
print(f"FP16: True")

=== 학습 설정 ===
Batch size: 64
Epochs: 3
Total steps: 1,599
Warmup steps: 159
Loss: MultipleNegativesRankingLoss
FP16: True


In [13]:
# 파인튜닝 실행
import os
os.environ["WANDB_DISABLED"] = "true"  # ⭐ WandB 비활성화

import logging
logging.basicConfig(
    format='%(asctime)s - %(message)s',
    datefmt='%H:%M:%S',
    level=logging.INFO
)

print("=== 파인튜닝 시작 ===")

model.fit(
    train_objectives=[(train_dataloader, train_loss)],
    evaluator=evaluator,
    epochs=NUM_EPOCHS,
    warmup_steps=warmup_steps,
    output_path=str(OUTPUT_PATH / 'finetuned'),
    evaluation_steps=500,
    save_best_model=True,
    show_progress_bar=True
)

print("\n✅ 파인튜닝 완료!")

=== 파인튜닝 시작 ===


Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).
Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).


Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

Step,Training Loss,Validation Loss,Qa-eval Pearson Cosine,Qa-eval Spearman Cosine
500,2.068900,No log,0.739668,0.732750
533,2.068900,No log,0.752509,0.744517
1000,1.484000,No log,0.758735,0.746117
1066,1.484000,No log,0.763193,0.749789
1500,1.309600,No log,0.763184,0.750773
1599,1.309600,No log,0.763946,0.751290


Batches:   0%|          | 0/250 [00:00<?, ?it/s]

...

Batches:   0%|          | 0/250 [00:00<?, ?it/s]

Batches:   0%|          | 0/250 [00:00<?, ?it/s]

Batches:   0%|          | 0/250 [00:00<?, ?it/s]

Batches:   0%|          | 0/250 [00:00<?, ?it/s]


✅ 파인튜닝 완료!


---
## 5. 파인튜닝 결과 평가

In [14]:
# 파인튜닝된 모델 로드
finetuned_model = SentenceTransformer(str(OUTPUT_PATH / 'finetuned'), device=device)

# Before vs After 비교
print("=== Before vs After Fine-tuning ===")
print(f"\n{'Query':<30} {'Before':>8} {'After':>8} {'Diff':>8}")
print("-" * 60)

original_model = SentenceTransformer(BASE_MODEL, device=device)
after_scores = []

for (q, a), before_score in zip(test_pairs, baseline_scores):
    # After
    emb_q = finetuned_model.encode(q, convert_to_tensor=True)
    emb_a = finetuned_model.encode(a, convert_to_tensor=True)
    after_score = torch.cosine_similarity(emb_q.unsqueeze(0), emb_a.unsqueeze(0)).item()
    after_scores.append(after_score)
    
    diff = after_score - before_score
    print(f"{q[:28]:<30} {before_score:>8.3f} {after_score:>8.3f} {diff:>+8.3f}")

print(f"\n평균: {np.mean(baseline_scores):.3f} → {np.mean(after_scores):.3f} ({np.mean(after_scores)-np.mean(baseline_scores):+.3f})")

=== Before vs After Fine-tuning ===

Query                            Before    After     Diff
------------------------------------------------------------
인터넷뱅킹 로그인이 안돼요                    0.579    0.586   +0.007
비밀번호 5회 오류                        0.692    0.827   +0.135
카드 분실 신고                          0.625    0.728   +0.103
코로나 검사 장소                         0.504    0.560   +0.056

평균: 0.600 → 0.675 (+0.075)


In [15]:
# 검색 성능 테스트
print("=== 유사 답변 검색 테스트 ===")

# 테스트용 답변 DB 구축
test_sample = val_qa.sample(n=1000, random_state=42)
test_questions = test_sample['question'].tolist()
test_answers = test_sample['answer'].tolist()
test_domains = test_sample['domain'].tolist()

# 답변 임베딩
print("답변 임베딩 생성 중...")
answer_embeddings = finetuned_model.encode(test_answers, show_progress_bar=True)

def search_answer(query, top_k=3):
    """질문으로 유사 답변 검색"""
    query_emb = finetuned_model.encode(query)
    sims = cosine_similarity([query_emb], answer_embeddings)[0]
    top_idx = np.argsort(sims)[::-1][:top_k]
    
    return [
        {
            'question': test_questions[i],
            'answer': test_answers[i],
            'domain': test_domains[i],
            'similarity': sims[i]
        }
        for i in top_idx
    ]

=== 유사 답변 검색 테스트 ===
답변 임베딩 생성 중...


Batches:   0%|          | 0/32 [00:00<?, ?it/s]

In [16]:
# 검색 테스트
test_queries = [
    "인터넷뱅킹 비밀번호 오류",
    "택배 배송 언제 오나요",
    "코로나 검사 어디서 받나요",
    "카드 한도 증액 방법"
]

for query in test_queries:
    print(f"\n{'='*70}")
    print(f"Query: {query}")
    print(f"{'='*70}")
    
    results = search_answer(query, top_k=3)
    for i, r in enumerate(results, 1):
        print(f"\n{i}. [유사도: {r['similarity']:.3f}] [{r['domain']}]")
        print(f"   Q: {r['question'][:60]}")
        print(f"   A: {r['answer'][:60]}")


Query: 인터넷뱅킹 비밀번호 오류

1. [유사도: 0.600] [금융/보험]
   Q: 안녕하세요. 인터넷 뱅킹 설치 중에 잘 안되서 문의드립니다.
   A: PC에서 플러그인 설치 중에 자꾸 오류가 납니다.

2. [유사도: 0.592] [금융/보험]
   Q: 기업인터넷 뱅킹의 공인인중서 암호 오류가 납니다. 어떻게 해야 하나요?
   A: 3회 오류시 접근이 제한됩니다..

3. [유사도: 0.566] [금융/보험]
   Q: 기업공인인증서 암호 분실시 어떻게 하나요?
   A: 공인인증서는 오류횟수 제한이 없습니다.

Query: 택배 배송 언제 오나요

1. [유사도: 0.435] [K쇼핑]
   Q: 네, 방송 하는 엘에이 갈비 주문 하려고요.
   A: 안녕하십니까? 쇼핑입니다. 무엇을 도와드릴까요?

2. [유사도: 0.426] [K쇼핑]
   Q: 주문 한 거 조회 좀 해주세요.
   A: 안녕하십니까? 쇼핑 입니다. 무엇을 도와드릴까요?

3. [유사도: 0.426] [K쇼핑]
   Q: 네, 사이즈 교환 가능합니까?
   A: 안녕하십니까? 쇼핑 입니다. 무엇을 도와드릴까요?

Query: 코로나 검사 어디서 받나요

1. [유사도: 0.739] [질병관리본부]
   Q: 코로나 의심증상이 있어서요. 검사는 어디서 받을 수 있나요?
   A: 코로나 선별진료소에서 검사 받을 수 있습니다.

2. [유사도: 0.731] [질병관리본부]
   Q: 코로나 의심증상이 있어서요. 검사는 어디서 하면되나요?
   A: 코로나 선별진료소에서 검사 받으실 수 있습니다.

3. [유사도: 0.511] [다산콜센터]
   Q: 재난문자가 왔는데요.
   A: 네. 그렇습니다. 발열 등의 증상이 나타나면 진료소 방문해주시면 됩니다.

Query: 카드 한도 증액 방법

1. [유사도: 0.478] [금융/보험]
   Q: 00카드에 포인트가 있어요
   A: 네 고객님 00카드입니다.

2. [유사도: 0.450] [금융/보험]
   Q: 이체한도를 

---
## 6. 모델 저장

In [17]:
# 최종 모델 저장
final_path = OUTPUT_PATH / 'civil_complaint_embedding'
finetuned_model.save(str(final_path))

# 설정 저장
config = {
    'base_model': BASE_MODEL,
    'embedding_dim': finetuned_model.get_sentence_embedding_dimension(),
    'num_train_pairs': len(train_pairs),
    'epochs': NUM_EPOCHS,
    'batch_size': BATCH_SIZE,
    'loss': 'MultipleNegativesRankingLoss',
    'scheduler': 'warmupcosine',
    'fp16': True,
    'data_source': 'qa_pairs.parquet',
    'baseline_avg_similarity': float(np.mean(baseline_scores)),
    'finetuned_avg_similarity': float(np.mean(after_scores))
}

with open(final_path / 'training_config.json', 'w') as f:
    json.dump(config, f, indent=2, ensure_ascii=False)

print(f"✅ 모델 저장 완료: {final_path}")
print(f"\n저장된 파일:")
for f in final_path.iterdir():
    print(f"  {f.name}")

✅ 모델 저장 완료: /kaggle/working/models/embedding/civil_complaint_embedding

저장된 파일:
  model.safetensors
  tokenizer_config.json
  tokenizer.json
  sentence_bert_config.json
  1_Pooling
  modules.json
  config.json
  vocab.txt
  training_config.json
  special_tokens_map.json
  README.md
  config_sentence_transformers.json


In [18]:
# 다운로드용 압축
!zip -r /kaggle/working/embedding_model.zip {final_path}
print("\n✅ 압축 완료: /kaggle/working/embedding_model.zip")

  adding: kaggle/working/models/embedding/civil_complaint_embedding/ (stored 0%)
  adding: kaggle/working/models/embedding/civil_complaint_embedding/model.safetensors

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


 (deflated 8%)
  adding: kaggle/working/models/embedding/civil_complaint_embedding/tokenizer_config.json (deflated 74%)
  adding: kaggle/working/models/embedding/civil_complaint_embedding/tokenizer.json (deflated 69%)
  adding: kaggle/working/models/embedding/civil_complaint_embedding/sentence_bert_config.json (deflated 9%)
  adding: kaggle/working/models/embedding/civil_complaint_embedding/1_Pooling/ (stored 0%)
  adding: kaggle/working/models/embedding/civil_complaint_embedding/1_Pooling/config.json (deflated 59%)
  adding: kaggle/working/models/embedding/civil_complaint_embedding/modules.json (deflated 53%)
  adding: kaggle/working/models/embedding/civil_complaint_embedding/config.json (deflated 50%)
  adding: kaggle/working/models/embedding/civil_complaint_embedding/vocab.txt (deflated 49%)
  adding: kaggle/working/models/embedding/civil_complaint_embedding/training_config.json (deflated 33%)
  adding: kaggle/working/models/embedding/civil_complaint_embedding/special_tokens_map.jso

---
## 7. 결과 요약

In [19]:
print("="*60)
print("04. 임베딩 모델 학습 결과 요약")
print("="*60)
print(f"\n[모델]")
print(f"  베이스: {BASE_MODEL}")
print(f"  임베딩 차원: {finetuned_model.get_sentence_embedding_dimension()}")
print(f"\n[학습]")
print(f"  데이터: {len(train_pairs):,}건")
print(f"  Epochs: {NUM_EPOCHS}")
print(f"  Batch: {BATCH_SIZE}")
print(f"  Loss: MultipleNegativesRankingLoss")
print(f"\n[성능 변화]")
print(f"  Before: {np.mean(baseline_scores):.3f}")
print(f"  After:  {np.mean(after_scores):.3f}")
print(f"  개선:   {np.mean(after_scores)-np.mean(baseline_scores):+.3f}")
print(f"\n[저장 위치]")
print(f"  {final_path}")

04. 임베딩 모델 학습 결과 요약

[모델]
  베이스: jhgan/ko-sroberta-multitask
  임베딩 차원: 768

[학습]
  데이터: 34,057건
  Epochs: 3
  Batch: 64
  Loss: MultipleNegativesRankingLoss

[성능 변화]
  Before: 0.600
  After:  0.675
  개선:   +0.075

[저장 위치]
  /kaggle/working/models/embedding/civil_complaint_embedding


---
## 8. 사용 방법 (로컬)

```python
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

# 모델 로드
model = SentenceTransformer('./models/embedding/civil_complaint_embedding')

# 질문 임베딩
query = "인터넷뱅킹 로그인 오류"
query_emb = model.encode(query)

# 답변 DB 임베딩 (미리 계산해두면 빠름)
answer_embs = model.encode(answer_list)

# 유사도 계산 및 Top-K 검색
similarities = cosine_similarity([query_emb], answer_embs)[0]
top_indices = similarities.argsort()[::-1][:5]

for idx in top_indices:
    print(f"[{similarities[idx]:.3f}] {answer_list[idx]}")
```

---
## 다음 단계

→ **05_rag_vectordb.ipynb** (로컬):
1. 파인튜닝된 임베딩 모델로 벡터 DB 구축 (ChromaDB)
2. RAG 체인 구성 (LangChain)
3. LLM 연동 (Gemini API / Ollama)